# Modern Physics for Electrical Engineers
**Quantum mechanics → semiconductors → photonics → coherent systems**

Coverage maps to Serway *Modern Physics* (your `ModPhy-Serway.pdf`) with EE emphasis.  
Runs in **Google Colab** or locally.

| § | Topic | EE Connection |
|---|-------|---------------|
| 1 | Planck / photoelectric effect | Photodetectors, solar cells |
| 2 | de Broglie waves + uncertainty | Transistor scaling limits |
| 3 | Schrödinger equation + wells | Quantum wells, LEDs, lasers |
| 4 | Hydrogen atom + orbitals | Spectroscopy, atomic clocks |
| 5 | Band theory + semiconductors | MOSFETs, diodes |
| 6 | Photon statistics + coherence | Coherent detection, LiDAR |
| 7 | Laser physics | Optical comms, sensing |
| 8 | Special relativity (EM fields) | Maxwell → Lorentz invariance |

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('pip install -q numpy matplotlib scipy sympy')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.constants import h, hbar, c, e, m_e, k, epsilon_0
from scipy.linalg import eigh
from sympy import *

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
print(f'h  = {h:.4e} J·s')
print(f'ħ  = {hbar:.4e} J·s')
print(f'c  = {c:.4e} m/s')
print(f'eV = {e:.4e} J')
print(f'mₑ = {m_e:.4e} kg')

---
## §1 · Planck radiation + photoelectric effect

$$B_\nu(T) = \frac{2h\nu^3}{c^2}\frac{1}{e^{h\nu/k_BT}-1} \qquad \text{(spectral radiance)}$$

Einstein (1905): photon energy $E = h\nu$.  Photoelectric threshold $\nu_0 = \phi/h$.

**EE link:** photodetector responsivity $\mathcal{R} = \eta e / (h\nu)$ — quantum efficiency $\eta$ is the fraction of photons that produce electron-hole pairs.

In [ ]:
nu = np.logspace(12, 15.5, 2000)  # Hz: microwave → UV

def planck(nu, T):
    x = h * nu / (k * T)
    return 2 * h * nu**3 / c**2 / (np.exp(np.clip(x, 0, 700)) - 1 + 1e-300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

temps = [1000, 3000, 5778, 10000]
for T in temps:
    axes[0].loglog(nu / 1e12, planck(nu, T), label=f'T = {T} K')
axes[0].axvline(c / 1550e-9 / 1e12, ls='--', color='gray', alpha=0.7, label='1550 nm (telecom)')
axes[0].axvline(c / 400e-9  / 1e12, ls=':', color='violet', alpha=0.7, label='400 nm (violet)')
axes[0].set_xlabel('Frequency (THz)'); axes[0].set_ylabel('Spectral radiance')
axes[0].set_title('Planck blackbody spectrum'); axes[0].legend(fontsize=8)

# Photoelectric: stopping voltage vs frequency
phi_eV = {'Cs': 2.1, 'Al': 4.1, 'Au': 5.1}  # work functions in eV
nu_range = np.linspace(0.5e15, 1.5e15, 300)
for metal, phi in phi_eV.items():
    phi_J = phi * e
    nu0 = phi_J / h
    Vs = np.maximum(0, h * (nu_range - nu0) / e)
    axes[1].plot(nu_range / 1e15, Vs, label=f'{metal} (φ={phi} eV)')
axes[1].set_xlabel('Frequency (PHz)'); axes[1].set_ylabel('Stopping voltage (V)')
axes[1].set_title('Photoelectric effect — stopping voltage'); axes[1].legend()

plt.tight_layout(); plt.savefig('mp_planck.png', bbox_inches='tight'); plt.show()

# Detector responsivity
lam_nm = np.linspace(400, 1700, 300)
nu_det = c / (lam_nm * 1e-9)
for eta in [0.5, 0.8, 1.0]:
    R = eta * e / (h * nu_det)
    plt.plot(lam_nm, R, label=f'η = {eta}')
plt.xlabel('Wavelength (nm)'); plt.ylabel('Responsivity R (A/W)')
plt.title('Ideal photodetector responsivity  R = ηe/hν'); plt.legend()
plt.axvline(1310, ls='--', color='gray', alpha=0.6, label='1310 nm O-band')
plt.axvline(1550, ls=':', color='orange', alpha=0.6)
plt.tight_layout(); plt.savefig('mp_responsivity.png', bbox_inches='tight'); plt.show()

## §2 · de Broglie waves + Heisenberg uncertainty

$$\lambda_{dB} = \frac{h}{p} = \frac{h}{mv} \qquad \Delta x \cdot \Delta p \geq \frac{\hbar}{2}$$

**EE link:** transistor gate length $L_g$ must stay $\gg \lambda_{dB}$ of channel electrons or quantum tunneling dominates leakage current.  At 3 nm node, $\lambda_{dB} \sim 1$ nm — we are at the wall.

In [ ]:
# de Broglie wavelength vs kinetic energy
KE_eV = np.logspace(-3, 5, 500)
KE_J  = KE_eV * e
v     = np.sqrt(2 * KE_J / m_e)
p     = m_e * v
lam   = h / p * 1e9  # nm

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(KE_eV, lam)
for gate_nm, label in [(3, '3 nm node'), (7, '7 nm'), (28, '28 nm'), (250, '250 nm')]:
    axes[0].axhline(gate_nm, ls='--', alpha=0.6, label=label)
axes[0].set_xlabel('Kinetic energy (eV)'); axes[0].set_ylabel('de Broglie λ (nm)')
axes[0].set_title('Electron de Broglie wavelength vs KE'); axes[0].legend(fontsize=8)

# Heisenberg: uncertainty in momentum vs position uncertainty
dx_nm = np.logspace(-1, 3, 300)
dp_min = hbar / 2 / (dx_nm * 1e-9)  # kg·m/s
dE_min = dp_min**2 / (2 * m_e) / e  # eV
axes[1].loglog(dx_nm, dE_min)
axes[1].axvline(3,  ls='--', color='red',   alpha=0.7, label='3 nm gate')
axes[1].axvline(28, ls='--', color='green', alpha=0.7, label='28 nm gate')
axes[1].set_xlabel('Position uncertainty Δx (nm)')
axes[1].set_ylabel('Min KE uncertainty (eV)')
axes[1].set_title('Heisenberg: quantum confinement energy floor'); axes[1].legend()

plt.tight_layout(); plt.savefig('mp_debroglie.png', bbox_inches='tight'); plt.show()

## §3 · Schrödinger equation — infinite + finite square wells

$$-\frac{\hbar^2}{2m}\frac{d^2\psi}{dx^2} + V(x)\psi = E\psi$$

**EE link:** quantum well lasers (GaAs/AlGaAs) confine carriers in a ~10 nm potential well.  Quantized energy levels → discrete emission wavelengths → tunable laser diodes.

In [ ]:
def solve_finite_well(V0_eV, L_nm, N=1000, m_eff=0.067):
    """
    Numerically solve finite square well via matrix diagonalization.
    m_eff: effective mass ratio (0.067 = GaAs conduction band)
    """
    m_star = m_eff * m_e
    L = L_nm * 1e-9
    V0 = V0_eV * e
    x = np.linspace(-2*L, 2*L, N)
    dx = x[1] - x[0]

    # Kinetic energy: finite-difference Laplacian
    T = -hbar**2 / (2 * m_star * dx**2) * (
        np.diag(-2 * np.ones(N)) +
        np.diag(np.ones(N-1), 1) +
        np.diag(np.ones(N-1), -1)
    )
    # Potential: flat walls
    V_x = np.where(np.abs(x) <= L/2, 0.0, V0)
    H = T + np.diag(V_x)

    evals, evecs = eigh(H, subset_by_index=[0, 5])
    return x, evals / e, evecs, V_x / e  # energies in eV

x, energies, wavefuncs, V_x = solve_finite_well(V0_eV=0.3, L_nm=10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x_nm = x * 1e9
axes[0].plot(x_nm, V_x, 'k', lw=2, label='V(x)')
for n in range(3):
    En = energies[n]
    psi = wavefuncs[:, n]
    psi_norm = psi / np.max(np.abs(psi)) * 0.04
    axes[0].plot(x_nm, psi_norm + En, label=f'n={n+1}, E={En*1000:.1f} meV')
    axes[0].axhline(En, ls='--', alpha=0.4)
axes[0].set_xlabel('Position (nm)'); axes[0].set_ylabel('Energy (eV)')
axes[0].set_title('Finite square well — GaAs QW (V₀=0.3 eV, L=10 nm)')
axes[0].legend(fontsize=9); axes[0].set_ylim(-0.02, 0.35)

# Emission wavelength vs well width
widths = np.linspace(2, 20, 50)
lam_emission = []
for L_nm in widths:
    _, En, _, _ = solve_finite_well(0.3, L_nm)
    delta_E = En[1] - En[0] if En[1] > 0 else En[0]
    Eg_GaAs_eV = 1.42
    E_photon = Eg_GaAs_eV + max(En[0], 0)
    lam_emission.append(h * c / (E_photon * e) * 1e9)
axes[1].plot(widths, lam_emission, 'o-', color='#e41a1c', markersize=4)
axes[1].axhline(870, ls='--', color='gray', label='Bulk GaAs (870 nm)')
axes[1].set_xlabel('Well width L (nm)'); axes[1].set_ylabel('Emission wavelength (nm)')
axes[1].set_title('QW laser: tuning emission via well width'); axes[1].legend()

plt.tight_layout(); plt.savefig('mp_quantum_well.png', bbox_inches='tight'); plt.show()
print(f'Bound state energies (eV): {[f"{e:.4f}" for e in energies[:4]]}')

## §4 · Hydrogen atom — energy levels + spectroscopy

$$E_n = -\frac{13.6\text{ eV}}{n^2}, \quad r_n = n^2 a_0, \quad a_0 = 0.529\text{ Å}$$

**EE link:** atomic clocks (Cs hyperfine at 9.192 GHz) define the SI second.  Optical atomic clocks use $n=1\to 2$ or similar transitions — accuracy $10^{-18}$ s/s.

In [ ]:
n_levels = np.arange(1, 9)
E_n = -13.6 / n_levels**2  # eV
a0 = 0.529e-10  # Bohr radius

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Energy level diagram
for n, En in zip(n_levels, E_n):
    axes[0].axhline(En, xmin=0.1, xmax=0.9, color='steelblue', lw=1.5)
    axes[0].text(0.92, En, f'n={n}  {En:.2f} eV', va='center', fontsize=9)

# Lyman, Balmer, Paschen series
series = [('Lyman', 1, 'UV', '#984ea3'), ('Balmer', 2, 'Vis', '#4daf4a'), ('Paschen', 3, 'IR', '#e41a1c')]
for name, n_low, band, col in series:
    for n_high in range(n_low+1, min(n_low+5, 9)):
        E_low, E_high = E_n[n_low-1], E_n[n_high-1]
        lam_nm = h * c / ((E_high - E_low) * e) * 1e9
        axes[0].annotate('', xy=(0.5, E_high), xytext=(0.5, E_low),
                         arrowprops=dict(arrowstyle='->', color=col, lw=1.2))
    print(f'{name} series ({band})')

axes[0].set_xlim(0, 1.2); axes[0].set_ylabel('Energy (eV)')
axes[0].set_title('Hydrogen energy levels'); axes[0].set_xticks([])
axes[0].axhline(0, color='k', lw=0.5, ls='--', alpha=0.4)
axes[0].text(0.5, 0.1, 'Ionization', ha='center', fontsize=9, color='gray')

# Emission spectrum — Balmer series
balmer_lam, balmer_intensity = [], []
for n in range(3, 8):
    E_photon = 13.6 * (1/4 - 1/n**2)
    lam = h * c / (E_photon * e) * 1e9
    balmer_lam.append(lam)
    balmer_intensity.append(1.0 / n**3)

lam_plot = np.linspace(380, 720, 2000)
spectrum = np.zeros_like(lam_plot)
for lam, I in zip(balmer_lam, balmer_intensity):
    spectrum += I * np.exp(-0.5 * ((lam_plot - lam)/1.5)**2)

# Color by wavelength
def wavelength_to_rgb(lam_nm):
    if   380 <= lam_nm < 440: r,g,b = (440-lam_nm)/60, 0, 1
    elif 440 <= lam_nm < 490: r,g,b = 0, (lam_nm-440)/50, 1
    elif 490 <= lam_nm < 510: r,g,b = 0, 1, (510-lam_nm)/20
    elif 510 <= lam_nm < 580: r,g,b = (lam_nm-510)/70, 1, 0
    elif 580 <= lam_nm < 645: r,g,b = 1, (645-lam_nm)/65, 0
    elif 645 <= lam_nm <= 720: r,g,b = 1, 0, 0
    else: r,g,b = 0,0,0
    return (r, g, b)

for i, (lam, s) in enumerate(zip(lam_plot, spectrum)):
    if s > 0.01:
        axes[1].axvline(lam, color=wavelength_to_rgb(lam), alpha=float(s), lw=1)

axes[1].set_xlabel('Wavelength (nm)'); axes[1].set_ylabel('Relative intensity')
axes[1].set_title('Hydrogen Balmer series emission spectrum')
axes[1].plot(lam_plot, spectrum, 'k', lw=0.8, alpha=0.4)
for lam, n in zip(balmer_lam, range(3, 8)):
    axes[1].text(lam, max(balmer_intensity)*0.05, f'n={n}→2\n{lam:.0f}nm',
                 ha='center', fontsize=8, rotation=90)

plt.tight_layout(); plt.savefig('mp_hydrogen.png', bbox_inches='tight'); plt.show()

## §5 · Band theory — semiconductors + doping

$$n_i = \sqrt{N_c N_v}\exp\!\left(-\frac{E_g}{2k_BT}\right), \quad n \cdot p = n_i^2$$

**EE link:** intrinsic carrier concentration $n_i$ sets leakage floor in MOSFETs and sets reverse-bias dark current in photodiodes.

In [ ]:
T_range = np.linspace(200, 600, 400)  # K

# Material parameters
materials = {
    'Si':   {'Eg0': 1.12, 'Nc': 2.8e25, 'Nv': 1.04e25, 'color': '#377eb8'},
    'Ge':   {'Eg0': 0.66, 'Nc': 1.04e25, 'Nv': 6.0e24,  'color': '#e41a1c'},
    'GaAs': {'Eg0': 1.42, 'Nc': 4.7e23, 'Nv': 7.0e24,  'color': '#4daf4a'},
    'InP':  {'Eg0': 1.35, 'Nc': 5.7e23, 'Nv': 1.1e25,  'color': '#984ea3'},
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, m in materials.items():
    ni = np.sqrt(m['Nc'] * m['Nv']) * np.exp(-m['Eg0'] * e / (2 * k * T_range))
    axes[0].semilogy(T_range, ni / 1e6, label=name, color=m['color'], lw=2)  # cm⁻³

axes[0].axvline(300, ls='--', color='gray', label='T=300 K')
axes[0].set_xlabel('Temperature (K)'); axes[0].set_ylabel('$n_i$ (cm⁻³)')
axes[0].set_title('Intrinsic carrier concentration vs T'); axes[0].legend()

# Fermi level position vs doping (Si at 300 K)
Nd_range = np.logspace(14, 20, 300)  # donors cm⁻³
ni_Si = 1.5e10  # cm⁻³ at 300 K
T0 = 300
# n-type: n ≈ Nd >> ni; Ef - Ei = kT * ln(Nd/ni)
dEf = k * T0 / e * np.log(Nd_range / ni_Si)  # eV above midgap
axes[1].semilogx(Nd_range, dEf, lw=2, color='#377eb8', label='n-type (Si, 300K)')
axes[1].semilogx(Nd_range, -dEf, lw=2, color='#e41a1c', ls='--', label='p-type (Na)')
axes[1].axhline(0.56, ls=':', color='gray', label='E_c (Si Eg/2 = 0.56 eV)')
axes[1].axhline(-0.56, ls=':', color='gray')
axes[1].set_xlabel('Dopant concentration (cm⁻³)')
axes[1].set_ylabel('$E_F - E_i$ (eV)')
axes[1].set_title('Fermi level shift with doping (Si)'); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('mp_bands.png', bbox_inches='tight'); plt.show()

## §6 · Photon statistics + coherence

Three light sources, three statistics:

| Source | Statistics | $g^{(2)}(0)$ | Example |
|--------|-----------|--------------|--------|
| Thermal | Bose-Einstein | 2 | LED, blackbody |
| Coherent | Poisson | 1 | Ideal laser |
| Single photon | Sub-Poisson | 0 | Quantum emitter |

**EE link:** coherent optical communications rely on Poisson (shot-noise limited) photon statistics.  The shot noise floor sets the fundamental SNR limit.

In [ ]:
from scipy.stats import poisson

mean_n = 10  # mean photon number
n_vals = np.arange(0, 35)

# Poisson (coherent laser)
P_poisson = poisson.pmf(n_vals, mean_n)

# Bose-Einstein (thermal)
P_BE = (mean_n / (1 + mean_n))**n_vals / (1 + mean_n)

# Sub-Poisson (approximate: binomial with high N)
from scipy.stats import binom
P_sub = binom.pmf(n_vals, 50, mean_n/50)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(n_vals - 0.3, P_poisson, width=0.28, label=f'Poisson (laser) σ²=μ={mean_n}', alpha=0.8, color='#377eb8')
axes[0].bar(n_vals,       P_BE,      width=0.28, label=f'Bose-Einstein (thermal) σ²>μ', alpha=0.8, color='#e41a1c')
axes[0].bar(n_vals + 0.3, P_sub,     width=0.28, label=f'Sub-Poisson (quantum) σ²<μ',  alpha=0.8, color='#4daf4a')
axes[0].set_xlabel('Photon number n'); axes[0].set_ylabel('Probability P(n)')
axes[0].set_title(f'Photon number distributions  ⟨n⟩={mean_n}'); axes[0].legend(fontsize=9)

# Shot noise limit for optical receiver
P_opt_dBm = np.linspace(-40, 0, 300)
P_opt_W   = 1e-3 * 10**(P_opt_dBm / 10)
lam_comm  = 1550e-9
R_det     = 0.9  # A/W responsivity
I_ph      = R_det * P_opt_W
BW        = 10e9  # 10 GHz
I_shot    = np.sqrt(2 * e * I_ph * BW)
SNR_shot  = 10 * np.log10((I_ph**2) / (I_shot**2 + 1e-50))

axes[1].plot(P_opt_dBm, SNR_shot, lw=2, color='crimson', label='Shot-noise limited SNR')
axes[1].axhline(13, ls='--', color='gray', label='13 dB (BER=10⁻³ QPSK)')
axes[1].set_xlabel('Received optical power (dBm)')
axes[1].set_ylabel('SNR (dB)')
axes[1].set_title('Coherent receiver: shot-noise SNR vs power (10 GHz BW)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('mp_photon_stats.png', bbox_inches='tight'); plt.show()

## §7 · Laser physics — threshold + rate equations

$$\frac{dN}{dt} = \frac{I}{eV} - \frac{N}{\tau_s} - v_g g S \qquad \frac{dS}{dt} = \Gamma v_g g S - \frac{S}{\tau_p} + \beta\frac{N}{\tau_s}$$

$N$ = carrier density, $S$ = photon density, $g$ = material gain, $\Gamma$ = confinement factor.

In [ ]:
from scipy.integrate import solve_ivp

# Laser diode parameters (InGaAsP, 1550 nm)
params = dict(
    tau_s  = 2e-9,      # carrier lifetime (s)
    tau_p  = 2e-12,     # photon lifetime (s)
    vg     = 8.3e9,     # group velocity (m/s)
    g0     = 1.5e-12,   # gain coefficient
    N_tr   = 1.5e24,    # transparency carrier density (m⁻³)
    Gamma  = 0.3,       # confinement factor
    beta   = 1e-4,      # spontaneous emission factor
    V      = 1e-16,     # active volume (m³)
)

def rate_eqs(t, y, I_A, p):
    N, S = y
    g = p['g0'] * (N - p['N_tr'])  # linear gain
    dN = I_A / (e * p['V']) - N / p['tau_s'] - p['vg'] * g * S
    dS = p['Gamma'] * p['vg'] * g * S - S / p['tau_p'] + p['beta'] * N / p['tau_s']
    return [dN, dS]

# Simulate step-on current at t=0
I_th_approx = e * params['V'] * params['N_tr'] / params['tau_s']  # rough threshold
I_bias = 3 * I_th_approx

sol = solve_ivp(rate_eqs, [0, 8e-9], [1e22, 1e10],
                args=(I_bias, params), max_step=1e-12,
                dense_output=True, method='Radau')

t_ns = sol.t * 1e9
N_sol, S_sol = sol.y

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].semilogy(t_ns, S_sol, color='#e41a1c', lw=1.5)
axes[0].set_ylabel('Photon density S (m⁻³)'); axes[0].set_title('Laser diode turn-on dynamics (rate equations)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_ns, N_sol / 1e24, color='#377eb8', lw=1.5)
axes[1].axhline(params['N_tr'] / 1e24, ls='--', color='gray', label='N_tr (transparency)')
axes[1].set_xlabel('Time (ns)'); axes[1].set_ylabel('Carrier density N (×10²⁴ m⁻³)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('mp_laser_rate.png', bbox_inches='tight'); plt.show()
print(f'Approximate threshold current: {I_th_approx*1e3:.2f} mA')

## §8 · Special relativity — fields + Lorentz invariance

$$E^2 = (pc)^2 + (m_0 c^2)^2 \qquad F^{\mu\nu} = \partial^\mu A^\nu - \partial^\nu A^\mu$$

**EE link:** Maxwell's equations are Lorentz-invariant — electromagnetism *is* a relativistic theory.  The magnetic force between currents is the relativistic correction to Coulomb repulsion.

In [ ]:
# E-p relation: compare relativistic vs classical
p_vals = np.logspace(-5, 2, 400)  # in units of m_e*c
p_SI   = p_vals * m_e * c

E_rel = np.sqrt((p_SI * c)**2 + (m_e * c**2)**2) / e / 1e6  # MeV
E_cls = (p_SI**2) / (2 * m_e) / e / 1e6 + m_e * c**2 / e / 1e6  # MeV, rest energy included

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(p_vals, E_rel, lw=2, label='Relativistic $E = \\sqrt{(pc)^2+(m_0c^2)^2}$')
axes[0].loglog(p_vals, E_cls, '--', lw=2, label='Classical $E = p^2/2m + m_0c^2$')
axes[0].set_xlabel('Momentum (units of $m_e c$)')
axes[0].set_ylabel('Total energy (MeV)')
axes[0].set_title('Relativistic vs classical dispersion relation')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Time dilation + length contraction
beta_vals = np.linspace(0, 0.999, 500)
gamma = 1 / np.sqrt(1 - beta_vals**2 + 1e-15)
axes[1].plot(beta_vals, gamma, 'crimson', lw=2, label='$\\gamma = (1-\\beta^2)^{-1/2}$')
axes[1].plot(beta_vals, 1/gamma, 'steelblue', lw=2, label='$1/\\gamma$ (length contraction)')
for b, lab in [(0.5,'0.5c'), (0.9,'0.9c'), (0.99,'0.99c')]:
    g = 1/np.sqrt(1-b**2)
    axes[1].axvline(b, ls=':', alpha=0.5)
    axes[1].text(b+0.005, 5, f'γ={g:.1f}', fontsize=8)
axes[1].set_xlabel('β = v/c'); axes[1].set_ylabel('γ  /  1/γ')
axes[1].set_title('Lorentz factor — time dilation & length contraction')
axes[1].set_ylim(0, 15); axes[1].legend()

plt.tight_layout(); plt.savefig('mp_relativity.png', bbox_inches='tight'); plt.show()

## Summary

| Concept | Formula | EE Application |
|---------|---------|----------------|
| Photon energy | $E=h\nu$ | Photodetector responsivity |
| de Broglie | $\lambda=h/p$ | Transistor scaling wall (~3 nm) |
| Quantum well | $H\psi = E\psi$ | QW laser wavelength tuning |
| Hydrogen levels | $E_n = -13.6/n^2$ eV | Atomic clock transitions |
| Band theory | $n_i \propto e^{-Eg/2kT}$ | MOSFET leakage floor |
| Photon stats | Poisson for laser | Shot-noise SNR limit |
| Rate equations | $dN/dt, dS/dt$ | Laser turn-on, modulation BW |
| Relativity | $E^2=(pc)^2+(m_0c^2)^2$ | EM field Lorentz invariance |